# Phase 2: Linear Mixed Model (LMM) Analysis

This notebook implements Linear Mixed Models (LMMs) to analyze sleep patterns, serving as a precursor to Generalized Additive Models (GAMs). 

**Objectives:**
1.  Load processed sleep data.
2.  Implement harmonic regression (sine/cosine terms) to capture seasonality.
3.  Run LMMs for Sleep Onset, Offset, and Midpoint using **circular outcomes** (sine and cosine components).
4.  Run LMM for Sleep Duration (linear outcome).
5.  Assess the impact of Age, Sex, Employment, and Weekend status while controlling for individual variability.

In [ ]:
# Load necessary libraries
library(arrow)
library(tidyverse)
library(lme4)
library(lmerTest)  # For p-values
library(gtsummary)
library(performance)

# Set seed for reproducibility
set.seed(123)

## 1. Load Data

In [ ]:
# Load the processed data
df <- arrow::read_parquet("processed_data/ready_for_analysis.parquet")

# Preview data
head(df)

## 2. Data Preparation & Feature Engineering

We need to ensure categorical variables are factors and create the seasonality terms using harmonic regression.
We also verify that the circular outcome variables (`onset_sin`, `onset_cos`, etc.) are present.

In [ ]:
# Data Cleaning and Feature Engineering
df_clean <- df %>%
  filter(!is.na(age_at_sleep), !is.na(sex_concept), !is.na(employment_status)) %>%
  mutate(
    # Ensure factors
    person_id = as.factor(person_id),
    sex_concept = as.factor(sex_concept),
    employment_status = as.factor(employment_status),
    is_weekend = as.logical(is_weekend),
    
    # Seasonality: Harmonic Regression Terms (Predictors)
    # day_of_year ranges from 1 to 365 (or 366)
    # We use a period of 365.25 days
    sin_doy = sin(2 * pi * day_of_year / 365.25),
    cos_doy = cos(2 * pi * day_of_year / 365.25)
  )

# Check structure and verify circular outcomes exist
str(df_clean %>%
      select(person_id, onset_sin, onset_cos, offset_sin, offset_cos, midpoint_sin, midpoint_cos))

## 3. Linear Mixed Models

We will fit LMMs with `person_id` as a random intercept.
For circular variables (Onset, Offset, Midpoint), we model the sine and cosine components separately.

**Model Structure:**
`Outcome_Component ~ Age + Sex + Employment + Weekend + sin(Year) + cos(Year) + (1|Person)`

### Model A: Sleep Onset (Bedtime)
Targets: `onset_sin`, `onset_cos`

In [ ]:
# Onset Sine Component
model_onset_sin <- lmer(
  onset_sin ~ age_at_sleep + sex_concept + employment_status + is_weekend + 
              sin_doy + cos_doy + (1 | person_id),
  data = df_clean
)

# Onset Cosine Component
model_onset_cos <- lmer(
  onset_cos ~ age_at_sleep + sex_concept + employment_status + is_weekend + 
              sin_doy + cos_doy + (1 | person_id),
  data = df_clean
)

# Summaries
tbl_merge(
  tbls = list(tbl_regression(model_onset_sin), tbl_regression(model_onset_cos)),
  tab_spanner = c("**Sine Component**", "**Cosine Component**")
)

### Model B: Sleep Offset (Wake Time)
Targets: `offset_sin`, `offset_cos`

In [ ]:
# Offset Sine Component
model_offset_sin <- lmer(
  offset_sin ~ age_at_sleep + sex_concept + employment_status + is_weekend + 
               sin_doy + cos_doy + (1 | person_id),
  data = df_clean
)

# Offset Cosine Component
model_offset_cos <- lmer(
  offset_cos ~ age_at_sleep + sex_concept + employment_status + is_weekend + 
               sin_doy + cos_doy + (1 | person_id),
  data = df_clean
)

# Summaries
tbl_merge(
  tbls = list(tbl_regression(model_offset_sin), tbl_regression(model_offset_cos)),
  tab_spanner = c("**Sine Component**", "**Cosine Component**")
)

### Model C: Sleep Midpoint (Chronotype)
Targets: `midpoint_sin`, `midpoint_cos`

In [ ]:
# Midpoint Sine Component
model_midpoint_sin <- lmer(
  midpoint_sin ~ age_at_sleep + sex_concept + employment_status + is_weekend + 
                 sin_doy + cos_doy + (1 | person_id),
  data = df_clean
)

# Midpoint Cosine Component
model_midpoint_cos <- lmer(
  midpoint_cos ~ age_at_sleep + sex_concept + employment_status + is_weekend + 
                 sin_doy + cos_doy + (1 | person_id),
  data = df_clean
)

# Summaries
tbl_merge(
  tbls = list(tbl_regression(model_midpoint_sin), tbl_regression(model_midpoint_cos)),
  tab_spanner = c("**Sine Component**", "**Cosine Component**")
)

### Model D: Sleep Duration
Target: `daily_duration_mins` (Linear)

In [ ]:
model_duration <- lmer(
  daily_duration_mins ~ age_at_sleep + sex_concept + employment_status + is_weekend + 
                        sin_doy + cos_doy + (1 | person_id),
  data = df_clean
)

# Summary
summary(model_duration)

# Formatted Table
tbl_regression(model_duration) %>%
  bold_p() %>%
  add_glance_source_note()

## 4. Diagnostics & Checks

Quick check of model performance and assumptions.

In [ ]:
# Check for collinearity (using one of the models)
check_collinearity(model_midpoint_sin)

# Visual check of residuals
plot(model_midpoint_sin)